<a href="https://colab.research.google.com/github/mehakrana16/infosys-maritime-ai/blob/main/Login.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.6 MB/s eta 0:00:00


In [6]:
%%writefile db.py
import sqlite3, bcrypt, datetime, time
from contextlib import contextmanager

DB_NAME = "auth_users.db"
MAX_ATTEMPTS = 5            # wrong tries allowed before the first lockout
BASE_LOCK_SECONDS = 60      # first lockout = 1 minute
MAX_LOCK_SECONDS = 3600     # lockout never grows beyond 1 hour
STALE_SECONDS = 86400       # old failures are forgotten after 24 hours

SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What was the name of your first pet?",
    "What is your mother's maiden name?",
    "In which city were you born?",
    "What is the name of your favourite teacher?",
]

@contextmanager
def _db():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()

def _now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

# --- Hashing helpers (bcrypt) ---
def hash_text(text):
    return bcrypt.hashpw(text.encode("utf-8")[:72], bcrypt.gensalt()).decode("utf-8")

def check_text(text, hashed):
    try:
        return bool(hashed) and bcrypt.checkpw(text.encode("utf-8")[:72], hashed.encode("utf-8"))
    except ValueError:
        return False

# --- Database setup ---
def init_db():
    with _db() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE COLLATE NOCASE,
            email TEXT NOT NULL UNIQUE COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL)""")
        c.execute("""CREATE TABLE IF NOT EXISTS lockouts (
            key TEXT PRIMARY KEY,
            fails INTEGER NOT NULL,
            locked_until REAL NOT NULL DEFAULT 0,
            last_fail REAL NOT NULL)""")

# --- Signup ---
def register_user(username, email, password, question, answer):
    username, email = username.strip(), email.strip().lower()
    with _db() as c:
        if c.execute("SELECT 1 FROM users WHERE username = ?", (username,)).fetchone():
            return False, "Username already taken. Please choose another one."
        if c.execute("SELECT 1 FROM users WHERE email = ?", (email,)).fetchone():
            return False, "This email is already registered."
        try:
            c.execute(
                "INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, created_at) "
                "VALUES (?, ?, ?, ?, ?, ?)",
                (username, email, hash_text(password), question, hash_text(answer.strip().lower()), _now()))
        except sqlite3.IntegrityError:
            return False, "Username or email already exists."
    return True, "Account created successfully."

# --- Lookups ---
def find_user(identifier):
    """Find a user by username OR email."""
    identifier = identifier.strip()
    with _db() as c:
        row = c.execute("SELECT * FROM users WHERE username = ? OR email = ?", (identifier, identifier)).fetchone()
    return dict(row) if row else None

def get_user_by_username(username):
    with _db() as c:
        row = c.execute("SELECT * FROM users WHERE username = ?", (username.strip(),)).fetchone()
    return dict(row) if row else None

def email_exists(email):
    with _db() as c:
        return c.execute("SELECT 1 FROM users WHERE email = ?", (email.strip().lower(),)).fetchone() is not None

# --- Login ---
def authenticate_user(identifier, password):
    user = find_user(identifier)
    if user and check_text(password, user["password_hash"]):
        return user
    return None

# --- Forgot password (security question route) ---
def get_security_question(username):
    user = get_user_by_username(username)
    return user["security_question"] if user else None

def verify_security_answer(username, answer):
    user = get_user_by_username(username)
    return bool(user) and check_text(answer.strip().lower(), user["security_answer_hash"])

def update_password(identifier, new_password):
    with _db() as c:
        c.execute("UPDATE users SET password_hash = ? WHERE username = ? OR email = ?",
                  (hash_text(new_password), identifier.strip(), identifier.strip()))

# --- Rate limiting with growing lockouts ---
# 5 wrong tries -> locked 1 min. After the lock ends, every further wrong try
# locks again for 1 minute longer (2 min, 3 min, 4 min ...). Success resets it.
def _fails(row, now):
    if not row or now - row["last_fail"] > STALE_SECONDS:
        return 0
    return row["fails"]

def failed_attempt(key):
    """Record one wrong try. Returns (total_fails, lock_seconds_started_now)."""
    key, now = key.lower(), time.time()
    with _db() as c:
        row = c.execute("SELECT fails, locked_until, last_fail FROM lockouts WHERE key = ?", (key,)).fetchone()
        fails = _fails(row, now) + 1
        lock_seconds, locked_until = 0, 0
        if fails >= MAX_ATTEMPTS:
            lock_seconds = min(BASE_LOCK_SECONDS * (fails - MAX_ATTEMPTS + 1), MAX_LOCK_SECONDS)
            locked_until = now + lock_seconds
        c.execute("INSERT OR REPLACE INTO lockouts (key, fails, locked_until, last_fail) VALUES (?, ?, ?, ?)",
                  (key, fails, locked_until, now))
    return fails, lock_seconds

def attempts_left(key):
    now = time.time()
    with _db() as c:
        row = c.execute("SELECT fails, locked_until, last_fail FROM lockouts WHERE key = ?", (key.lower(),)).fetchone()
    return max(0, MAX_ATTEMPTS - _fails(row, now))

def lockout_remaining(key):
    """Seconds left in the current lockout (0 = not locked)."""
    now = time.time()
    with _db() as c:
        row = c.execute("SELECT locked_until FROM lockouts WHERE key = ?", (key.lower(),)).fetchone()
    if not row:
        return 0
    return max(0, int(row["locked_until"] - now) + (1 if row["locked_until"] > now else 0))

def clear_attempts(key):
    with _db() as c:
        c.execute("DELETE FROM lockouts WHERE key = ?", (key.lower(),))

# --- Admin: list users (never returns passwords or answers) ---
def get_all_users():
    with _db() as c:
        rows = c.execute("SELECT username, email, created_at FROM users ORDER BY id").fetchall()
    return [dict(r) for r in rows]


Overwriting db.py


In [3]:
%%writefile assets.py
# Embedded images: faded cube backdrop (pink, light/dark) and the Infosys Springboard panel image
CUBES = {"light": {"l": "..."}, "dark": {"l": "..."}}
INFOSYS_B64 = "..."
# Infosys Springboard panel image with the cream background removed (transparent WebP) so it blends into the pink theme
PROMO_WEBP_B64 = "..."


Writing assets.py


In [7]:
%%writefile app.py
import os, re, html, time, secrets, smtplib, datetime
import jwt
import streamlit as st
import streamlit.components.v1 as components
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import db, assets

PROJECT_TITLE = "Agentic AI for Maritime Freight Pricing and Route Optimization"
PROJECT_SUBTITLE = "Milestone 1 · User Authentication Module"

st.set_page_config(page_title="Secure Authentication Portal", page_icon="🌸", layout="centered")

# ── Secrets (the launcher cell copies them from Colab Secrets into the environment) ──
JWT_SECRET = os.getenv("JWT_SECRET")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
ADMIN_USERNAME = "admin"                                   # admin login is defined in code (Milestone rule)
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "Admin@12345")
OTP_EXPIRY_MINUTES = 5
SESSION_MINUTES = 30

# ── Themes: text is dark on the light theme and light on the dark theme ──
LIGHT = {
    "bg": "#FFFAFC", "bg2": "#FFF1F6", "card": "#FFFFFF", "card-alt": "#FDF3F7",
    "text": "#3D3036", "muted": "#7A6770", "primary": "#F8C8DC", "secondary": "#F4B6C2",
    "accent": "#D98BA3", "lavender": "#EDE4F7", "input": "#FFFFFF", "border": "#F4B6C2",
    "shadow": "rgba(217,139,163,0.20)", "hero-from": "#F8C8DC", "hero-to": "#EDE4F7",
    "hero-text": "#3D3036", "btn": "#F8C8DC", "btn-text": "#3D3036", "btn-hover": "#D98BA3",
    "orb1": "#F8C8DC", "orb2": "#EDE4F7",
    "err-bg": "#FFE8EC", "err-border": "#F2A1B1", "ok-bg": "#E6F6EC", "ok-border": "#9AD4B0",
    "info-bg": "#EDE4F7", "info-border": "#C9B6E8", "warn-bg": "#FFF3DC", "warn-border": "#F0CE8A",
}
DARK = {
    "bg": "#171019", "bg2": "#22152A", "card": "#271B2D", "card-alt": "#31233A",
    "text": "#F7ECF1", "muted": "#C3ADB9", "primary": "#E79AB8", "secondary": "#5A3A4D",
    "accent": "#F29CBB", "lavender": "#3A2D4D", "input": "#33243B", "border": "#5A3A4D",
    "shadow": "rgba(0,0,0,0.45)", "hero-from": "#5B2A4E", "hero-to": "#33255A",
    "hero-text": "#FFF0F6", "btn": "#F29CBB", "btn-text": "#2A1420", "btn-hover": "#FFB8D0",
    "orb1": "#7A2F5C", "orb2": "#3B2C7A",
    "err-bg": "#43202E", "err-border": "#8F3B58", "ok-bg": "#1F3D2E", "ok-border": "#41805E",
    "info-bg": "#2E2548", "info-border": "#5D4C90", "warn-bg": "#40341F", "warn-border": "#8D7236",
}
LIGHT["link"] = "#B4527A"
DARK["link"] = "#FFB8D0"
dark_mode = bool(st.session_state.get("dark_mode", False))
P = DARK if dark_mode else LIGHT

BASE_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700&family=Inter:wght@400;500;600&display=swap');
html{scroll-behavior:smooth}
html,body,.stApp{font-family:'Inter',sans-serif !important}
.stApp{background:linear-gradient(180deg,var(--bg) 0%,var(--bg2) 100%) !important;transition:background .4s ease}
header[data-testid="stHeader"],footer,#MainMenu,[data-testid="stToolbar"]{display:none !important}
.stApp::before,.stApp::after{content:"";position:fixed;border-radius:50%;filter:blur(90px);z-index:0;pointer-events:none}
.stApp::before{top:-150px;right:-130px;width:460px;height:460px;background:var(--orb1);opacity:.55;animation:drift1 16s ease-in-out infinite alternate}
.stApp::after{bottom:-170px;left:-150px;width:480px;height:480px;background:var(--orb2);opacity:.7;animation:drift2 18s ease-in-out infinite alternate}
@keyframes drift1{to{transform:translate(-50px,70px) scale(1.08)}}
@keyframes drift2{to{transform:translate(60px,-50px) scale(1.1)}}
@keyframes fadeUp{from{opacity:0;transform:translateY(14px)}to{opacity:1;transform:translateY(0)}}
@keyframes shimmer{0%{background-position:0% 50%}50%{background-position:100% 50%}100%{background-position:0% 50%}}
.block-container{position:relative;z-index:1;max-width:880px !important;padding:1.6rem 1.5rem 2.5rem !important}

/* Text colours follow the theme */
.stApp, .stApp p, .stApp li, .stApp label, .stApp h1, .stApp h2, .stApp h3, .stApp h4,
.stApp [data-testid="stMarkdownContainer"], .stApp [data-testid="stWidgetLabel"] p {color:var(--text) !important}
.stApp [data-testid="stCaptionContainer"], .stApp small {color:var(--muted) !important}
.stApp h1,.stApp h2,.stApp h3,.stApp h4{font-family:'Poppins',sans-serif !important}
:focus-visible{outline:3px solid var(--accent) !important;outline-offset:2px}

/* Hero banner */
.stApp .hero{background:linear-gradient(135deg,var(--hero-from),var(--hero-to),var(--hero-from));background-size:200% 200%;
  animation:shimmer 12s ease infinite,fadeUp .7s ease both;border:1.5px solid var(--border);border-radius:24px;
  padding:26px 30px;margin:.2rem 0 1.2rem;box-shadow:0 14px 34px var(--shadow)}
.stApp .hero .tag{font-size:12px;font-weight:700;letter-spacing:1.6px;color:var(--hero-text) !important;opacity:.8}
.stApp .hero h1{color:var(--hero-text) !important;font-size:1.85rem !important;line-height:1.25 !important;margin:8px 0 6px !important;padding:0 !important;font-weight:700 !important}
.stApp .hero p{color:var(--hero-text) !important;opacity:.85;margin:0;font-size:14px}

/* Tabs */
div[data-baseweb="tab-list"]{gap:6px}
button[data-baseweb="tab"]{border-radius:10px 10px 0 0;transition:background .25s ease}
button[data-baseweb="tab"]:hover{background:var(--card-alt) !important}
button[data-baseweb="tab"] p{color:var(--muted) !important;font-weight:600 !important}
button[data-baseweb="tab"][aria-selected="true"] p{color:var(--text) !important}
div[data-baseweb="tab-highlight"]{background-color:var(--accent) !important;height:3px !important;border-radius:3px}
div[data-baseweb="tab-border"]{background-color:var(--border) !important}
div[data-baseweb="tab-panel"]{animation:fadeUp .5s ease both;padding-top:1rem !important}
.tab-title{font-family:'Poppins',sans-serif;font-size:1.55rem;font-weight:700;color:var(--text);margin:0 0 4px}
.tab-sub{color:var(--muted);font-size:14.5px;margin:0 0 14px}

/* Cards (forms + bordered containers) */
div[data-testid="stForm"],div[data-testid="stVerticalBlockBorderWrapper"]{background:var(--card) !important;
  border:1.5px solid var(--border) !important;border-radius:20px !important;padding:22px !important;
  box-shadow:0 10px 28px var(--shadow) !important;transition:transform .3s ease,box-shadow .3s ease}
div[data-testid="stForm"]:hover,div[data-testid="stVerticalBlockBorderWrapper"]:hover{transform:translateY(-2px);box-shadow:0 16px 36px var(--shadow) !important}

/* Inputs */
div[data-baseweb="base-input"]{background:transparent !important;border:none !important}
div[data-baseweb="input"],div[data-baseweb="select"]>div{background:var(--input) !important;border:1.5px solid var(--border) !important;
  border-radius:12px !important;transition:all .25s ease}
div[data-baseweb="input"]:hover,div[data-baseweb="select"]>div:hover{border-color:var(--accent) !important}
div[data-baseweb="input"]:focus-within,div[data-baseweb="select"]>div:focus-within{border-color:var(--accent) !important;box-shadow:0 0 0 4px var(--shadow) !important}
input{background:transparent !important;color:var(--text) !important;-webkit-text-fill-color:var(--text) !important;font-size:16px !important}
input::placeholder{color:var(--muted) !important;-webkit-text-fill-color:var(--muted) !important;opacity:.75}
div[data-baseweb="select"] div,div[data-baseweb="select"] span{color:var(--text) !important}
div[data-baseweb="input"] svg,div[data-baseweb="select"] svg{fill:var(--text) !important;color:var(--text) !important}
div[data-baseweb="popover"] ul,div[data-baseweb="popover"] li,div[data-baseweb="popover"]>div{background:var(--card) !important;color:var(--text) !important}
div[data-baseweb="popover"] li:hover{background:var(--card-alt) !important}
div[data-testid="InputInstructions"],div[data-testid="InputInstructions"] *{color:var(--muted) !important}

/* Buttons */
div[data-testid="stButton"] button,div[data-testid="stFormSubmitButton"] button{background:var(--btn) !important;color:var(--btn-text) !important;
  border:1.5px solid var(--border) !important;border-radius:12px !important;font-weight:600 !important;font-size:15px !important;
  height:48px !important;width:100% !important;box-shadow:0 5px 14px var(--shadow) !important;transition:all .25s ease !important}
div[data-testid="stButton"] button p,div[data-testid="stFormSubmitButton"] button p{color:var(--btn-text) !important;font-weight:600 !important}
div[data-testid="stButton"] button:hover,div[data-testid="stFormSubmitButton"] button:hover{background:var(--btn-hover) !important;
  border-color:var(--btn-hover) !important;transform:translateY(-2px) !important;box-shadow:0 10px 22px var(--shadow) !important}

/* Messages */
.stApp .notice{display:flex;gap:12px;align-items:flex-start;padding:13px 16px;border-radius:14px;margin:12px 0;font-size:14.5px;
  color:var(--text) !important;animation:fadeUp .4s ease both;border:1.5px solid}
.stApp .notice div{color:var(--text) !important}
.notice-error{background:var(--err-bg);border-color:var(--err-border)}
.notice-success{background:var(--ok-bg);border-color:var(--ok-border)}
.notice-info{background:var(--info-bg);border-color:var(--info-border)}
.notice-warning{background:var(--warn-bg);border-color:var(--warn-border)}

/* Password strength meter */
.meter{display:flex;align-items:center;gap:12px;margin:2px 0 8px}
.meter-bar{flex:1;height:8px;border-radius:99px;background:var(--card-alt);border:1px solid var(--border);overflow:hidden}
.meter-fill{height:100%;border-radius:99px;transition:width .4s ease}
.meter-label{font-size:12.5px;font-weight:600;color:var(--muted);min-width:70px;text-align:right}
.chips{display:flex;flex-wrap:wrap;gap:6px;margin-bottom:10px}
.chip{font-size:12px;padding:3px 10px;border-radius:99px;border:1px solid var(--border);color:var(--muted);background:var(--card-alt);transition:all .3s ease}
.chip.ok{background:var(--ok-bg);border-color:var(--ok-border);color:var(--text)}

/* Dashboard */
.dash{display:flex;align-items:center;gap:18px;padding:24px;border-radius:22px;background:var(--card);border:1.5px solid var(--border);
  box-shadow:0 12px 30px var(--shadow);animation:fadeUp .6s ease both;margin-bottom:16px}
.avatar{width:66px;height:66px;border-radius:50%;background:linear-gradient(135deg,var(--accent),var(--primary));display:flex;align-items:center;
  justify-content:center;font-family:'Poppins',sans-serif;font-weight:700;font-size:28px;color:#2A1420}
.dash-hi{color:var(--muted);font-size:14px}
.dash-name{font-family:'Poppins',sans-serif;font-size:1.6rem;font-weight:700;color:var(--text)}
.dash-mail{color:var(--muted);font-size:14px}
.tiles{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:14px;margin-bottom:18px}
.tile{padding:18px;border-radius:18px;background:var(--card);border:1.5px solid var(--border);transition:all .3s ease;animation:fadeUp .7s ease both}
.tile:hover{transform:translateY(-4px);box-shadow:0 14px 28px var(--shadow)}
.tile-ic{font-size:22px;margin-bottom:6px}.tile-lb{font-size:12.5px;color:var(--muted)}
.tile-v{font-weight:700;font-size:16px;color:var(--text);margin-top:2px}
.tbl-wrap{overflow-x:auto;border-radius:16px;border:1.5px solid var(--border);background:var(--card);margin-bottom:16px}
.tbl{width:100%;border-collapse:collapse;font-size:14px}
.tbl th{text-align:left;padding:12px 14px;background:var(--card-alt);color:var(--muted);font-weight:600;border-bottom:1px solid var(--border)}
.tbl td{padding:12px 14px;color:var(--text);border-bottom:1px solid var(--border)}
.tbl tr:last-child td{border-bottom:none}.tbl tbody tr:hover td{background:var(--card-alt)}
.footer-note{text-align:center;color:var(--muted);font-size:13px;margin-top:2.2rem;padding-top:1rem;border-top:1px solid var(--border)}

@media (prefers-reduced-motion:reduce){*{animation:none !important;transition:none !important;scroll-behavior:auto !important}}
@media (max-width:700px){.stApp .hero h1{font-size:1.4rem !important}.dash{flex-direction:column;text-align:center}}
"""

NEW_CSS = """
/* ===== 1. FLUID / ADAPTIVE SIZING: follows zoom in/out and any screen size ===== */
@media (min-width:1400px){html{font-size:clamp(16px,.95vw,24px) !important}}
.block-container{max-width:min(66rem,100%) !important;width:100% !important;
  padding:clamp(.8rem,2vw,1.6rem) clamp(.7rem,2.4vw,1.5rem) 2.5rem !important}
.stApp .hero{padding:clamp(16px,2.4vw,26px) clamp(16px,2.8vw,30px) !important;border-radius:clamp(16px,2vw,24px) !important}
.stApp .hero h1{font-size:clamp(1.2rem,.85rem + 1.6vw,1.9rem) !important;overflow-wrap:anywhere}
.stApp .hero .tag{font-size:clamp(10px,.5vw + 7px,12px) !important}
.st-key-auth_card{background:var(--card) !important;border:1.5px solid var(--border) !important;border-radius:clamp(16px,2vw,22px) !important;
  padding:clamp(16px,2.4vw,26px) clamp(16px,2.6vw,28px) !important;box-shadow:0 12px 30px var(--shadow) !important}
.st-key-auth_card div[data-testid="stForm"]{background:transparent !important;border:none !important;box-shadow:none !important;padding:0 !important;transform:none !important}
.st-key-auth_card div[data-testid="stForm"]:hover{transform:none !important;box-shadow:none !important}
.tab-title{font-size:clamp(1.25rem,1rem + 1vw,1.55rem) !important}

/* the small rows (dark-mode toggle, banner + close button, remember/trouble row) never wrap */
div[data-testid="stHorizontalBlock"]:has(.st-key-dark_mode){flex-wrap:nowrap !important;align-items:center !important}
div[data-testid="stHorizontalBlock"]:has(.st-key-dark_mode)>div:first-child{flex:1 1 0 !important;min-width:0 !important}
div[data-testid="stHorizontalBlock"]:has(.st-key-dark_mode)>div:last-child{flex:0 0 auto !important;width:auto !important;min-width:max-content !important}
div[data-testid="stHorizontalBlock"]:has(.st-key-banner_x){flex-wrap:nowrap !important;align-items:flex-start !important;gap:.6rem !important}
div[data-testid="stHorizontalBlock"]:has(.st-key-banner_x)>div:first-child{flex:1 1 0 !important;min-width:0 !important;width:auto !important}
div[data-testid="stHorizontalBlock"]:has(.st-key-banner_x)>div:last-child{flex:0 0 3rem !important;width:3rem !important;min-width:3rem !important}
.stApp .notice{margin-top:0}
.st-key-banner_x button{height:40px !important;padding:0 !important}
div[data-testid="stForm"] div[data-testid="stHorizontalBlock"]{flex-wrap:nowrap !important;align-items:center !important;gap:.5rem !important}
div[data-testid="stForm"] div[data-testid="stHorizontalBlock"]>div:first-child{flex:1 1 0 !important;min-width:0 !important;width:auto !important}
div[data-testid="stForm"] div[data-testid="stHorizontalBlock"]>div:last-child{flex:0 0 auto !important;width:auto !important;min-width:max-content !important}

/* ===== 2. INPUTS + CHECKBOX: readable in BOTH light and dark mode ===== */
div[data-testid="stTextInputRootElement"],div[data-testid="stTextInput"] div[data-baseweb="input"],
div[data-testid="stTextInput"] div[data-baseweb="base-input"],div[data-testid="stSelectbox"] div[data-baseweb="select"]>div{
  background:var(--input) !important;background-color:var(--input) !important}
div[data-testid="stTextInputRootElement"]{border:1.5px solid var(--border) !important;border-radius:12px !important;transition:all .25s ease}
div[data-testid="stTextInputRootElement"] div[data-baseweb="input"],div[data-testid="stTextInputRootElement"] div[data-baseweb="base-input"]{
  border:none !important;box-shadow:none !important}
div[data-testid="stTextInputRootElement"]:hover{border-color:var(--accent) !important}
div[data-testid="stTextInputRootElement"]:focus-within{border-color:var(--accent) !important;box-shadow:0 0 0 4px var(--shadow) !important}
input,textarea{background:transparent !important;color:var(--text) !important;-webkit-text-fill-color:var(--text) !important;caret-color:var(--text) !important}
input:-webkit-autofill,input:-webkit-autofill:hover,input:-webkit-autofill:focus{
  -webkit-box-shadow:0 0 0 1000px var(--input) inset !important;-webkit-text-fill-color:var(--text) !important;caret-color:var(--text) !important}
div[data-testid="stTextInputRootElement"] button{background:transparent !important;color:var(--text) !important;box-shadow:none !important}
div[data-testid="stTextInputRootElement"] button svg{fill:var(--text) !important;color:var(--text) !important}
[data-testid="stCheckbox"] label[data-baseweb="checkbox"]>span:first-child{background:var(--input) !important;border:2px solid var(--accent) !important;border-radius:6px !important}
[data-testid="stCheckbox"] label[data-baseweb="checkbox"]:has(input:checked)>span:first-child{background:var(--accent) !important}
[data-testid="stCheckbox"] label p,[data-testid="stRadio"] label p{color:var(--text) !important}

/* ===== 3. BUTTONS: AWS-style. Sign in + admin are rounded rectangles; the two links are plain text (no box) ===== */
div[data-testid="stButton"] button,div[data-testid="stFormSubmitButton"] button{border-radius:10px !important}
.st-key-hid_submit{display:none !important}                     /* invisible first submit button so Enter always means Sign in */
.st-key-btn_signin button p{font-weight:700 !important}
.stApp .st-key-sec_admin div[data-testid="stFormSubmitButton"] button,.stApp .st-key-sec_admin div[data-testid="stButton"] button{
  background:var(--card) !important;border:2px solid var(--accent) !important;box-shadow:none !important}
.stApp .st-key-sec_admin button p{color:var(--link) !important}
.stApp .st-key-sec_admin button:hover{background:var(--card-alt) !important;border-color:var(--link) !important}
.stApp [class*="st-key-lnk_"] div[data-testid="stButton"] button,.stApp [class*="st-key-lnk_"] div[data-testid="stFormSubmitButton"] button{
  background:transparent !important;border:none !important;box-shadow:none !important;height:auto !important;width:auto !important;
  min-height:0 !important;padding:2px 0 !important;border-radius:4px !important;transform:none !important}
.stApp [class*="st-key-lnk_"] button p{color:var(--link) !important;font-weight:600 !important;font-size:clamp(13px,.3vw + 11px,15px) !important;
  text-decoration:underline dotted;text-underline-offset:3px}
.stApp [class*="st-key-lnk_"] button:hover{background:transparent !important;transform:none !important;box-shadow:none !important}
.stApp [class*="st-key-lnk_"] button:hover p{text-decoration-style:solid}
.st-key-lnk_trouble div[data-testid="stFormSubmitButton"]{display:flex;justify-content:flex-end}
.st-key-lnk_signup div[data-testid="stFormSubmitButton"]{display:flex;justify-content:center}
.info-i{font-size:15px;color:var(--link);cursor:help;margin-left:6px;font-family:'Inter',sans-serif}
.legal{color:var(--muted);font-size:clamp(11.5px,.3vw + 9.5px,13px);line-height:1.55;margin:14px 6px 0}
.stApp .banner-title{font-weight:700;margin-bottom:2px}

/* ===== 4. INFOSYS PANEL: transparent image on a soft pink panel, same height as the sign-in card ===== */
.promo{position:relative;height:100%;min-height:clamp(20rem,34vw,32rem);box-sizing:border-box;overflow:hidden;
  border-radius:clamp(16px,2vw,22px);border:1.5px solid var(--border);box-shadow:0 12px 30px var(--shadow);
  background:linear-gradient(160deg,#FFF4F8 0%,#F9D6E4 55%,#EDE4F7 100%)}
.promo img{position:absolute;top:4%;left:4%;width:92%;height:92%;object-fit:contain;object-position:center}
@media (min-width:901px){
  div[data-testid="stHorizontalBlock"]:has(.promo){align-items:stretch !important}
  div[data-testid="stHorizontalBlock"]:has(.promo)>div:last-child,
  div[data-testid="stHorizontalBlock"]:has(.promo)>div:last-child :is(div[data-testid="stVerticalBlock"],div[data-testid="stElementContainer"],.element-container,div[data-testid="stMarkdown"],div[data-testid="stMarkdownContainer"]){height:100% !important}
}
/* tablets / phones / big zoom-in: sign-in card on top, image panel below */
@media (max-width:900px){
  div[data-testid="stHorizontalBlock"]:has(.st-key-auth_card){flex-wrap:wrap !important}
  div[data-testid="stHorizontalBlock"]:has(.st-key-auth_card)>div{flex:1 1 100% !important;width:100% !important;min-width:100% !important}
  .promo{height:auto;min-height:0;aspect-ratio:4/5;width:min(100%,26rem);margin:0 auto}
}
"""
def cube_css():
    m = "dark" if dark_mode else "light"
    L, R = assets.CUBES[m]["l"], assets.CUBES[m]["r"]
    size = "min(34vw,420px) auto no-repeat fixed"
    grad = "linear-gradient(180deg,var(--bg) 0%,var(--bg2) 100%)"
    return (".stApp{background:url(data:image/png;base64," + L + ") left top 10vh / " + size + ","
            "url(data:image/png;base64," + R + ") right bottom 2vh / " + size + "," + grad + " !important}"
            "@media (max-width:900px){.stApp{background:" + grad + " !important}}")
root_vars = ":root{color-scheme:" + ("dark" if dark_mode else "light") + ";" + "".join(f"--{k}:{v};" for k, v in P.items()) + "}"
st.markdown("<style>" + root_vars + BASE_CSS + NEW_CSS + cube_css() + "</style>", unsafe_allow_html=True)

# ── Small UI helpers ──
def notice(kind, text):
    icon = {"error": "⚠️", "success": "✅", "info": "ℹ️", "warning": "⏳"}[kind]
    st.markdown(f'<div class="notice notice-{kind}"><span>{icon}</span><div>{html.escape(text)}</div></div>', unsafe_allow_html=True)

def lock_banner(seconds, title="Too many wrong attempts"):
    """Live ticking countdown (runs in the browser)."""
    page = f"""
<div style="font-family:Inter,Arial,sans-serif;display:flex;align-items:center;gap:14px;padding:14px 18px;border-radius:16px;
background:{P['warn-bg']};border:1.5px solid {P['warn-border']};color:{P['text']}">
<div style="font-size:28px">🔒</div>
<div><div style="font-weight:700;font-size:15px">{html.escape(title)}</div>
<div style="font-size:14px;opacity:.9">Please wait <b id="t" style="font-size:18px">--:--</b> before trying again.</div></div></div>
<script>
let s={int(seconds)};const el=document.getElementById('t');
function f(){{const m=Math.floor(s/60),r=s%60;el.textContent=String(m).padStart(2,'0')+':'+String(r).padStart(2,'0');}}
f();
const i=setInterval(()=>{{if(s<=0){{clearInterval(i);el.textContent='00:00 - you can try again now';return;}}s--;f();}},1000);
</script>"""
    if hasattr(st, "iframe"):          # newer Streamlit
        st.iframe(page, height=92)
    else:                              # older Streamlit
        components.html(page, height=92)

def strength_meter(pw):
    checks = [("10+ characters", len(pw) >= 10), ("Uppercase", bool(re.search(r"[A-Z]", pw))),
              ("Lowercase", bool(re.search(r"[a-z]", pw))), ("Number", bool(re.search(r"\d", pw))),
              ("Symbol", bool(re.search(r"[^A-Za-z0-9]", pw)))]
    score = sum(ok for _, ok in checks)
    labels = ["Empty", "Very weak", "Weak", "Fair", "Good", "Strong"]
    colors = ["#cfcfcf", "#E57A8A", "#E5937A", "#E8B05B", "#9CCB7A", "#5CBF8B"]
    chips = "".join(f'<span class="chip{" ok" if ok else ""}">{"✓" if ok else "•"} {n}</span>' for n, ok in checks)
    return (f'<div class="meter"><div class="meter-bar"><div class="meter-fill" style="width:{score*20}%;background:{colors[score]}"></div></div>'
            f'<div class="meter-label">{labels[score]}</div></div><div class="chips">{chips}</div>')

# ── Validation ──
USERNAME_RE = re.compile(r"^[A-Za-z0-9_]{3,30}$")
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$")
COMMON = {"password@123", "welcome@123", "admin@123", "qwerty@123", "password123!", "admin@12345"}

def is_valid_email(email):
    """2+ letters before @, 2+ letters between @ and the last dot, 2+ letters after it."""
    if not EMAIL_RE.match(email):
        return False
    local, domain = email.split("@")
    host, tld = domain.rsplit(".", 1)
    letters = lambda s: sum(ch.isalpha() for ch in s)
    return letters(local) >= 2 and letters(host) >= 2 and len(tld) >= 2

def is_strong_password(pw, *names):
    if len(pw) < 10: return False, "Password must be at least 10 characters long."
    if len(pw) > 64: return False, "Password must be at most 64 characters long."
    if " " in pw: return False, "Password must not contain spaces."
    if not re.search(r"[A-Z]", pw): return False, "Add at least one uppercase letter."
    if not re.search(r"[a-z]", pw): return False, "Add at least one lowercase letter."
    if not re.search(r"\d", pw): return False, "Add at least one number."
    if not re.search(r"[^A-Za-z0-9]", pw): return False, "Add at least one special symbol (e.g. @, #, $, !)."
    for n in names:
        part = n.split("@")[0].lower()
        if len(part) >= 3 and part in pw.lower():
            return False, "Password must not contain your username or email name."
    if pw.lower() in COMMON: return False, "This password is too common."
    return True, "Strong"

# ── JWT: OTP tokens and login sessions ──
def _utcnow(): return datetime.datetime.now(datetime.timezone.utc)

def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    now = _utcnow()
    payload = {"sub": email, "otp_hash": db.hash_text(otp), "type": "password_reset_otp",
               "iat": now, "exp": now + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, code, email):
    try:
        p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.ExpiredSignatureError:
        return False, f"This OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please go back and request a new one.", True
    except jwt.InvalidTokenError:
        return False, "Invalid or corrupted verification token.", True
    if p.get("sub") != email or p.get("type") != "password_reset_otp":
        return False, "Security token mismatch.", True
    if db.check_text(code, p.get("otp_hash", "")):
        return True, "Valid", False
    return False, "Incorrect OTP code.", False

def make_session_token(username, role):
    now = _utcnow()
    payload = {"sub": username, "role": role, "type": "session", "iat": now,
               "exp": now + datetime.timedelta(minutes=SESSION_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def read_session_token(token):
    if not token: return None
    try:
        p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.InvalidTokenError:
        return None
    return p if p.get("type") == "session" else None

# ── Email (Gmail SMTP over SSL) ──
def send_otp_email(to_email, otp):
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Secure Portal <{EMAIL_ADDRESS}>"
    msg["To"] = to_email
    msg["Subject"] = "Your verification code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = EMAIL_ADDRESS
    text = f"Your verification code is: {otp}\nIt expires in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request it, ignore this email."
    page = f"""<html><body style="font-family:Arial,sans-serif;background:#FFFAFC;padding:20px">
<div style="max-width:480px;margin:0 auto;background:#fff;border:1.5px solid #F4B6C2;border-radius:16px;padding:30px;text-align:center">
<div style="font-size:20px;font-weight:bold;color:#3D3036;margin-bottom:12px">Password reset verification</div>
<div style="color:#5E4D55;font-size:15px;line-height:1.5">Use this code to reset the password for <b>{html.escape(to_email)}</b>:</div>
<div style="display:inline-block;margin:18px 0;padding:14px 22px;background:#F8C8DC;color:#3D3036;font-size:28px;font-weight:bold;letter-spacing:6px;border:1.5px solid #D98BA3;border-radius:12px">{otp}</div>
<div style="color:#5E4D55;font-size:14px">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
<div style="color:#7A6770;font-size:12px;margin-top:22px;border-top:1px solid #EDE4F7;padding-top:14px">If you did not request this, you can safely ignore this email.</div>
</div></body></html>"""
    msg.attach(MIMEText(text, "plain")); msg.attach(MIMEText(page, "html"))
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as s:
            s.login(EMAIL_ADDRESS, (EMAIL_PASSWORD or "").replace(" ", "").strip())
            s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        return True, "Email sent successfully!"
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail rejected the login. Check that EMAIL_PASSWORD is an App Password for EMAIL_ADDRESS."
    except Exception as e:
        return False, f"SMTP Error: {e}"

# ── Wrong-attempt handling (5 tries, then 1 min, 2 min, 3 min ...) ──
def blocked(key, label):
    """If this key is locked, show the countdown and return True."""
    left = db.lockout_remaining(key)
    if left:
        lock_banner(left, f"{label} is temporarily locked")
        return True
    return False

def wrong_attempt(key, message):
    fails, lock = db.failed_attempt(key)
    if lock:
        mins = lock // 60
        notice("error", f"{message} Too many wrong attempts. Locked for {mins} minute{'s' if mins != 1 else ''}.")
        lock_banner(lock, "Too many wrong attempts")
    else:
        notice("error", f"{message} Attempts left: {db.attempts_left(key)}.")

# ── Session handling ──
def logout():
    for k in list(st.session_state.keys()):
        if k not in ("dark_mode", "remembered_id", "banner_open"):
            del st.session_state[k]

def start_session(username, role):
    st.session_state.token = make_session_token(username, role)

# ── Dashboards ──
def user_dashboard(session):
    name = session["sub"]
    user = db.get_user_by_username(name) or {}
    mins = max(0, int((session["exp"] - time.time()) // 60))
    st.markdown(f'<div class="dash"><div class="avatar">{html.escape(name[:1].upper())}</div><div>'
                f'<div class="dash-hi">Welcome back 👋</div><div class="dash-name">{html.escape(name)}</div>'
                f'<div class="dash-mail">{html.escape(user.get("email", ""))}</div></div></div>', unsafe_allow_html=True)
    tiles = [("🟢", "Account status", "Active"), ("🔐", "Session", "JWT verified"),
             ("⏱️", "Session expires in", f"~{mins} min"), ("📅", "Member since", (user.get("created_at") or "")[:10])]
    st.markdown('<div class="tiles">' + "".join(
        f'<div class="tile"><div class="tile-ic">{i}</div><div class="tile-lb">{l}</div><div class="tile-v">{html.escape(v)}</div></div>'
        for i, l, v in tiles) + '</div>', unsafe_allow_html=True)
    st.button("Logout", key="logout_btn", on_click=logout)

def admin_dashboard(session):
    users = db.get_all_users()
    st.markdown('<div class="dash"><div class="avatar">🛡️</div><div><div class="dash-hi">Administrator</div>'
                '<div class="dash-name">Admin Dashboard</div><div class="dash-mail">All registered users (passwords are never shown)</div></div></div>',
                unsafe_allow_html=True)
    latest = users[-1]["username"] if users else "-"
    tiles = [("👥", "Registered users", str(len(users))), ("🆕", "Latest signup", latest), ("🔐", "Session", "JWT verified")]
    st.markdown('<div class="tiles">' + "".join(
        f'<div class="tile"><div class="tile-ic">{i}</div><div class="tile-lb">{l}</div><div class="tile-v">{html.escape(v)}</div></div>'
        for i, l, v in tiles) + '</div>', unsafe_allow_html=True)
    if users:
        rows = "".join(f"<tr><td>{n}</td><td>{html.escape(u['username'])}</td><td>{html.escape(u['email'])}</td><td>{html.escape(u['created_at'])}</td></tr>"
                       for n, u in enumerate(users, 1))
        st.markdown('<div class="tbl-wrap"><table class="tbl"><thead><tr><th>#</th><th>Username</th><th>Email</th><th>Created (UTC)</th></tr></thead>'
                    f'<tbody>{rows}</tbody></table></div>', unsafe_allow_html=True)
    else:
        notice("info", "No users have registered yet.")
    st.button("Logout", key="logout_btn", on_click=logout)

# ── NAVIGATION HELPERS (AWS-style: one card, links switch the view) ──
def go_view(view):
    st.session_state["view"] = view

def close_banner():
    st.session_state["banner_open"] = False

def kc(key):
    """Container with a CSS hook (st-key-<key>); falls back on very old Streamlit."""
    try:
        return st.container(key=key)
    except TypeError:
        return st.container()

def auth_card():
    try:
        return st.container(border=True, key="auth_card")
    except TypeError:                      # very old Streamlit without container keys
        return st.container(border=True)

# ── LOGIN TAB ──
def login_tab():
    st.markdown('<div class="tab-title">Sign in <span class="info-i" title="Use your username or email. Administrators use the Admin button below.">ⓘ</span></div>'
                '<div class="tab-sub">Log in with your username or email.</div>', unsafe_allow_html=True)
    if "li_id" not in st.session_state and st.session_state.get("remembered_id"):
        st.session_state["li_id"] = st.session_state["remembered_id"]
    with st.form("login_form"):
        with kc("hid_submit"):                      # hidden: it must be the FIRST submit button so pressing Enter = Sign in
            enter_go = st.form_submit_button("Submit")
        ident = st.text_input("Username or email", key="li_id", max_chars=60, placeholder="Enter your username or email")
        pw = st.text_input("Password", key="li_pw", type="password", max_chars=64, placeholder="Enter your password")
        c_left, c_right = st.columns(2)             # AWS-style row: checkbox on the left, help link on the right
        with c_left:
            remember = st.checkbox("Remember this account", key="li_rem")
        with c_right:
            with kc("lnk_trouble"):
                st.form_submit_button("Having trouble?", on_click=go_view, args=("forgot",))
        with kc("btn_signin"):
            go = st.form_submit_button("Sign in", type="primary", use_container_width=True)
        with kc("sec_admin"):
            st.form_submit_button("Sign in as administrator", on_click=go_view, args=("admin",), use_container_width=True)
        with kc("lnk_signup"):
            st.form_submit_button("Create a new account", on_click=go_view, args=("signup",))
    go = go or enter_go
    if go:
        ident = ident.strip()
        key = "login:" + ident.lower()
        if not ident or not pw:
            notice("error", "Please fill in all fields.")
        elif blocked(key, "Login"):
            pass
        else:
            user = db.authenticate_user(ident, pw)
            if user:
                db.clear_attempts(key)
                if remember: st.session_state["remembered_id"] = ident
                else: st.session_state.pop("remembered_id", None)
                start_session(user["username"], "user")
                st.rerun()
            else:
                wrong_attempt(key, "Invalid username/email or password.")

# ── SIGNUP TAB ──
def do_signup():
    s = st.session_state
    username, email = s.get("su_username", "").strip(), s.get("su_email", "").strip().lower()
    pw, cpw, q, ans = s.get("su_pw", ""), s.get("su_cpw", ""), s.get("su_q"), s.get("su_ans", "")
    def fail(m): s["su_msg"] = ("error", m)
    if not (username and email and pw and cpw and ans.strip()): return fail("All fields are mandatory.")
    if not USERNAME_RE.match(username): return fail("Username must be 3-30 characters: letters, numbers or underscore only.")
    if not is_valid_email(email): return fail("Enter a valid email (at least 2 letters before @, between @ and the dot, and after the dot).")
    ok, m = is_strong_password(pw, username, email)
    if not ok: return fail(m)
    if pw != cpw: return fail("Password and Confirm Password do not match.")
    if len(ans.strip()) < 2: return fail("Security answer is too short.")
    ok, m = db.register_user(username, email, pw, q, ans)
    if not ok: return fail(m)
    s["su_msg"] = ("success", "Account created successfully. Use Back to sign in below to log in.")
    for k in ("su_username", "su_email", "su_pw", "su_cpw", "su_ans"):
        s[k] = ""

def signup_tab():
    st.markdown('<div class="tab-title">Create your account</div><div class="tab-sub">Use a valid email and create a strong password.</div>', unsafe_allow_html=True)
    with st.container():
        st.text_input("Username", key="su_username", max_chars=30, placeholder="e.g. captain_nova")
        st.text_input("Email address", key="su_email", max_chars=60, placeholder="you@example.com")
        st.text_input("Password", key="su_pw", type="password", max_chars=64, placeholder="Create a strong password")
        st.markdown(strength_meter(st.session_state.get("su_pw", "")), unsafe_allow_html=True)
        st.text_input("Confirm password", key="su_cpw", type="password", max_chars=64, placeholder="Re-enter your password")
        st.selectbox("Security question", db.SECURITY_QUESTIONS, key="su_q")
        st.text_input("Security answer", key="su_ans", type="password", max_chars=50, placeholder="Your answer (used to recover your account)")
        st.button("Create account", key="su_btn", on_click=do_signup)
    if st.session_state.get("su_msg"):
        notice(*st.session_state["su_msg"])

# ── FORGOT PASSWORD TAB ──
def finish_reset(message, keys):
    for k in keys: st.session_state.pop(k, None)
    st.session_state["fp_msg"] = message
    st.rerun()

def new_password_fields(prefix):
    npw = st.text_input("New password", key=f"{prefix}_npw", type="password", max_chars=64, placeholder="Create a strong password")
    cpw = st.text_input("Confirm new password", key=f"{prefix}_cpw", type="password", max_chars=64, placeholder="Re-enter the new password")
    return npw, cpw

def security_question_flow():
    user = st.session_state.get("sq_user")
    if not user:
        with st.form("sq_find_form"):
            uname = st.text_input("Registered username", key="sq_uname", max_chars=30, placeholder="Enter your username")
            find = st.form_submit_button("Find security question")
        if find:
            q = db.get_security_question(uname) if uname.strip() else None
            if not uname.strip(): notice("error", "Please enter your username.")
            elif not q: notice("error", "Username not found.")
            else:
                st.session_state["sq_user"], st.session_state["sq_question"] = uname.strip(), q
                st.rerun()
        return
    key = "sq:" + user.lower()
    notice("info", f"Security question: {st.session_state['sq_question']}")
    blocked(key, "Security answer")
    with st.form("sq_reset_form"):
        ans = st.text_input("Security answer", key="sq_ans", type="password", max_chars=50, placeholder="Your answer")
        npw, cpw = new_password_fields("sq")
        go = st.form_submit_button("Reset password")
    if go:
        if not (ans.strip() and npw and cpw): notice("error", "All fields are mandatory.")
        elif db.lockout_remaining(key): pass   # countdown banner is already shown above
        elif not db.verify_security_answer(user, ans): wrong_attempt(key, "Incorrect security answer.")
        else:
            ok, m = is_strong_password(npw, user)
            if not ok: notice("error", m)
            elif npw != cpw: notice("error", "Password and Confirm Password do not match.")
            else:
                db.update_password(user, npw); db.clear_attempts(key)
                finish_reset("Password reset successful. Use Back to sign in below to log in.", ["sq_user", "sq_question", "sq_ans", "sq_npw", "sq_cpw", "sq_uname"])
    if st.button("← Start over", key="sq_back"):
        for k in ["sq_user", "sq_question", "sq_ans", "sq_npw", "sq_cpw", "sq_uname"]:
            st.session_state.pop(k, None)
        st.rerun()

def otp_flow():
    stage = st.session_state.get("otp_stage", "email")
    email = st.session_state.get("otp_email", "")
    all_keys = ["otp_stage", "otp_email", "otp_token", "otp_code", "otp_in", "otp_npw", "otp_cpw"]
    if stage == "email":
        with st.form("otp_email_form"):
            e = st.text_input("Registered email", key="otp_in", max_chars=60, placeholder="Enter your registered email")
            send = st.form_submit_button("Send verification code")
        if send:
            e = e.strip().lower()
            if not e: notice("error", "Please enter your email.")
            elif not is_valid_email(e): notice("error", "Enter a valid email address.")
            elif not (EMAIL_ADDRESS and EMAIL_PASSWORD): notice("error", "EMAIL_ADDRESS / EMAIL_PASSWORD are missing in Colab Secrets.")
            elif blocked("otp:" + e, "OTP verification"): pass
            elif not db.email_exists(e): notice("error", "Email address not registered in the system.")
            else:
                otp = generate_otp()
                with st.spinner("Sending your 6-digit code..."):
                    ok, m = send_otp_email(e, otp)
                if ok:
                    st.session_state.update(otp_stage="verify", otp_email=e, otp_token=make_otp_token(e, otp))
                    st.rerun()
                else:
                    notice("error", m)
    elif stage == "verify":
        key = "otp:" + email
        notice("info", f"Code sent to {email}. It is valid for {OTP_EXPIRY_MINUTES} minutes.")
        blocked(key, "OTP verification")
        with st.form("otp_verify_form"):
            code = st.text_input("6-digit verification code", key="otp_code", max_chars=6, placeholder="e.g. 849201")
            ok_btn = st.form_submit_button("Verify code")
        if ok_btn:
            if not (code.isdigit() and len(code) == 6): notice("error", "Please enter the 6-digit code.")
            elif db.lockout_remaining(key): pass   # countdown banner is already shown above
            else:
                ok, m, dead = verify_otp_token(st.session_state.get("otp_token", ""), code, email)
                if ok:
                    db.clear_attempts(key)
                    st.session_state["otp_stage"] = "reset"
                    st.rerun()
                elif dead: notice("error", m)
                else: wrong_attempt(key, m)
        if st.button("← Back", key="otp_back"):
            for k in all_keys: st.session_state.pop(k, None)
            st.rerun()
    elif stage == "reset":
        notice("success", "Code verified. Create your new password.")
        with st.form("otp_reset_form"):
            npw, cpw = new_password_fields("otp")
            go = st.form_submit_button("Reset password")
        if go:
            if not (npw and cpw): notice("error", "All fields are mandatory.")
            else:
                ok, m = is_strong_password(npw, email)
                if not ok: notice("error", m)
                elif npw != cpw: notice("error", "Password and Confirm Password do not match.")
                else:
                    db.update_password(email, npw)
                    finish_reset("Password reset successful. Use Back to sign in below to log in.", all_keys)

def forgot_tab():
    st.markdown('<div class="tab-title">Recover your account</div><div class="tab-sub">Reset using your security answer or an email verification code.</div>', unsafe_allow_html=True)
    msg = st.session_state.pop("fp_msg", None)
    if msg: notice("success", msg)
    method = st.radio("Recovery method", ["Security question", "Email OTP"], horizontal=True, key="fp_method")
    if method == "Security question": security_question_flow()
    else: otp_flow()

# ── ADMIN LOGIN TAB ──
def admin_tab():
    st.markdown('<div class="tab-title">Admin access</div><div class="tab-sub">Authorised administrators only.</div>', unsafe_allow_html=True)
    with st.form("admin_form"):
        u = st.text_input("Admin username", key="ad_user", max_chars=40, placeholder="Admin username")
        p = st.text_input("Admin password", key="ad_pw", type="password", max_chars=64, placeholder="Admin password")
        go = st.form_submit_button("Admin login")
    if go:
        if not u.strip() or not p: notice("error", "Please fill in all fields.")
        elif blocked("admin", "Admin login"): pass
        elif secrets.compare_digest(u.strip().encode(), ADMIN_USERNAME.encode()) and secrets.compare_digest(p.encode(), ADMIN_PASSWORD.encode()):
            db.clear_attempts("admin")
            start_session(ADMIN_USERNAME, "admin")
            st.rerun()
        else:
            wrong_attempt("admin", "Invalid admin credentials.")

# ═════════════ PAGE ═════════════
st.session_state.setdefault("view", "login")
_, toggle_col = st.columns([3, 1])
with toggle_col:
    getattr(st, "toggle", st.checkbox)("🌙 Dark mode", key="dark_mode")
st.markdown(f'<div class="hero"><div class="tag">⚓ AGENTIC AI PROJECT</div><h1>{html.escape(PROJECT_TITLE)}</h1><p>{html.escape(PROJECT_SUBTITLE)}</p></div>', unsafe_allow_html=True)

if not JWT_SECRET:
    notice("error", "JWT_SECRET is missing. Add it in Colab Secrets and run the launcher cell again.")
    st.stop()

db.init_db()
session = read_session_token(st.session_state.get("token"))
if st.session_state.get("token") and not session:
    st.session_state.pop("token", None)
    notice("warning", "Your session expired. Please log in again.")

if session:
    (admin_dashboard if session.get("role") == "admin" else user_dashboard)(session)
else:
    if st.session_state.get("banner_open", True):
        b1, b2 = st.columns([14, 1])
        with b1:
            st.markdown('<div class="notice notice-info"><span>ℹ️</span><div><div class="banner-title">Internship practice portal</div>'
                        'Do not use real sensitive accounts or production data on this page.</div></div>', unsafe_allow_html=True)
        with b2:
            st.button("✕", key="banner_x", on_click=close_banner, help="Dismiss")
    left, right = st.columns([1.1, 0.9], gap="large")
    with left:
        with auth_card():
            view = st.session_state.get("view", "login")
            {"login": login_tab, "signup": signup_tab, "forgot": forgot_tab, "admin": admin_tab}.get(view, login_tab)()
            if view not in ("login", "signup", "forgot", "admin"):
                st.session_state["view"] = "login"
            if view in ("signup", "forgot", "admin"):
                st.button("← Back to sign in", key="lnk_back", on_click=go_view, args=("login",))
    with right:
        st.markdown('<div class="promo"><img alt="Infosys Springboard - Empowering Learners, Shaping Futures" '
                    'src="data:image/webp;base64,' + assets.PROMO_WEBP_B64 + '"></div>', unsafe_allow_html=True)
    st.markdown('<div class="legal">By continuing, you confirm this is an internship practice portal and that you will not enter real sensitive data. '
                'This site uses only essential session cookies.</div>', unsafe_allow_html=True)

st.markdown('<div class="footer-note">© 2026 · Infosys Springboard Virtual Internship project</div>', unsafe_allow_html=True)


Overwriting app.py


In [ ]:
%pip install psutil python-dotenv pyngrok -q
import os, sys, time, subprocess
import psutil
from dotenv import load_dotenv

# 1. Load secrets: Colab Secrets (🔑 icon in the left sidebar) first, then an uploaded .env file as backup
NAMES = ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_PASSWORD", "NGROK_AUTHTOKEN"]
try:
    from google.colab import userdata
    for n in NAMES:
        try:
            v = userdata.get(n)
            if v: os.environ[n] = v
        except Exception:
            pass                      # secret not added, or "Notebook access" toggle is off
except ImportError:
    pass
if os.path.exists(".env"):
    load_dotenv(".env", override=False)   # only fills what Colab Secrets did not provide
    print("✅ Also read the uploaded .env file")
for n in NAMES:
    if not os.getenv(n):
        print(f"⚠️ Missing {n}  ->  add it in Colab Secrets (🔑) and turn ON 'Notebook access'")

# 2. Stop old Streamlit servers
for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if "streamlit" in cmd and "run" in cmd and p.pid != os.getpid():
            p.kill()
    except Exception:
        pass
time.sleep(2)

# 3. Start Streamlit in the background
process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                            "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                           env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 4. Open it: ngrok link if NGROK_AUTHTOKEN exists, otherwise Colab's own built-in proxy (no ngrok needed)
token = (os.getenv("NGROK_AUTHTOKEN") or "").strip()
if token:
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnels = ngrok.get_tunnels()
    public_url = tunnels[0].public_url if tunnels else ngrok.connect(8501).public_url
    print("=" * 65); print(f"👉 Authentication Portal Live URL: {public_url}"); print("=" * 65)
else:
    from google.colab import output
    print("ℹ️ No NGROK_AUTHTOKEN found - using Colab's built-in link instead. Click the link below:")
    output.serve_kernel_port_as_window(8501)
print("⏳ Server running. Click the Stop/Interrupt button to shut down.")
try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    process.kill()
    print("🛑 Streamlit stopped.")


👉 Authentication Portal Live URL: https://protector-runner-neatness.ngrok-free.dev
⏳ Server running. Click the Stop/Interrupt button to shut down.
